# Lab 02A: Agentic Workflow Patterns (Ollama Implementation)

This lab implements the five agentic workflow patterns described in Anthropic's engineering blog post [*Building Effective Agents*](https://www.anthropic.com/engineering/building-effective-agents).

All examples run **locally via Ollama** (zero API cost).

---

### Patterns covered

| # | Pattern | Core idea |
|---|---------|----------|
| 1 | **Prompt Chaining** | Sequential steps where each LLM call feeds the next |
| 2 | **Routing** | Classify input → dispatch to a specialised prompt/model |
| 3 | **Parallelization** | Run multiple LLM calls simultaneously, then aggregate |
| 4 | **Orchestrator-Workers** | Central LLM breaks down a task and delegates to workers |
| 5 | **Evaluator-Optimizer** | Generate → Evaluate → Refine loop |

> **Key insight from Anthropic:** Start simple. Only add complexity when it demonstrably improves outcomes.


## 0) Setup

Same environment as Lab 01. Make sure Ollama is running and your model is available.


In [35]:
import os
import asyncio
import json
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL    = os.getenv("OLLAMA_MODEL", "qwen3:8b")

client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL.rstrip('/')}/v1",
    api_key=os.getenv("OLLAMA_API_KEY", "ollama"),
)

def chat(messages: list[dict], temperature: float = 0.7) -> str:
    """Thin wrapper: returns the assistant reply as a plain string."""
    response = client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

print("Model:", OLLAMA_MODEL)
print("Ready ✓")


Model: qwen3:8b
Ready ✓


---
## Pattern 1 — Prompt Chaining

**Concept:** Decompose a complex task into a fixed sequence of smaller LLM calls. Each step's output becomes the next step's input. A *gate* (programmatic check) can be inserted between steps to catch problems early.

```
User Input → [Step 1 LLM] → output_1 → (gate?) → [Step 2 LLM] → output_2 → …
```

**Use when:** the task can be cleanly decomposed into sequential subtasks, and you want to trade latency for higher accuracy per step.

**Example below:** Blog post pipeline — outline → gate check → full draft → polish.


In [2]:
TOPIC = "Why vector databases matter for RAG applications"

# ── Step 1: Generate an outline ──────────────────────────────────────────────
outline = chat([
    {"role": "system", "content": "You are a technical content strategist."},
    {"role": "user",   "content": f"Write a 3-section outline for a blog post about: {TOPIC}. Return only the outline."},
])
display(Markdown("### Step 1 — Outline\n\n" + outline))

# ── Gate: check the outline has exactly 3 sections ───────────────────────────
section_count = outline.count("\n#") + outline.count("\n1.") + outline.count("\n2.")
gate_pass = len(outline.strip()) > 50   # simple length guard
print(f"\n🔍 Gate check — outline non-empty: {gate_pass}")
assert gate_pass, "Outline too short — halting pipeline."

# ── Step 2: Write the draft ───────────────────────────────────────────────────
draft = chat([
    {"role": "system",    "content": "You are a senior technical writer."},
    {"role": "user",      "content": f"Topic: {TOPIC}"},
    {"role": "assistant", "content": outline},
    {"role": "user",      "content": "Expand the outline into a concise 200-word blog post draft."},
])
display(Markdown("### Step 2 — Draft\n\n" + draft))

# ── Step 3: Polish for a non-technical audience ───────────────────────────────
polished = chat([
    {"role": "system", "content": "You are an editor who simplifies technical writing."},
    {"role": "user",   "content": f"Rewrite the following blog post so it is engaging for a non-technical audience. Keep it under 150 words.\n\n{draft}"},
])
display(Markdown("### Step 3 — Polished\n\n" + polished))


### Step 1 — Outline

**Outline: Why Vector Databases Matter for RAG Applications**  

1. **The Role of Vector Databases in Retrieval-Augmented Generation (RAG)**  
   - Overview of RAG and its reliance on semantic search.  
   - Challenges of traditional databases in handling unstructured data.  
   - Introduction to vector databases as a solution for efficient semantic similarity searches.  

2. **Key Advantages of Vector Databases in RAG Systems**  
   - Scalability and performance for large-scale embedding storage.  
   - Real-time query capabilities and reduced latency.  
   - Enhanced accuracy in retrieving contextually relevant information.  

3. **Use Cases and Future Impact of Vector Databases in RAG**  
   - Examples of RAG applications (e.g., customer support, content creation).  
   - How vector databases enable more dynamic and accurate responses.  
   - The growing importance of vector databases in AI-driven workflows.


🔍 Gate check — outline non-empty: True


### Step 2 — Draft

**Why Vector Databases Matter for RAG Applications**  

Retrieval-Augmented Generation (RAG) relies on semantic search to fetch relevant information from unstructured data, enabling AI models to generate accurate, context-aware responses. Traditional databases struggle with this task due to their inability to efficiently handle high-dimensional embeddings and semantic similarity queries. Vector databases, however, are designed to store and retrieve these embeddings with speed and precision, making them critical for RAG systems.  

Their key advantages include scalability for large-scale data, real-time query performance, and enhanced accuracy in matching user intent with relevant content. By leveraging efficient indexing and similarity search algorithms, vector databases reduce latency and ensure dynamic, contextually rich results.  

In practice, RAG applications like customer support chatbots, content creation tools, and personalized recommendation systems benefit immensely from vector databases. These systems enable faster, more accurate responses by bridging the gap between raw data and meaningful insights. As AI-driven workflows evolve, vector databases will remain a cornerstone for unlocking the full potential of RAG, driving innovation in semantic understanding and real-time decision-making.

### Step 3 — Polished

**Why Vector Databases Power Smart AI Tools**  

Imagine a chatbot that answers questions by sifting through vast data like a librarian finding the right book. Retrieval-Augmented Generation (RAG) does just that—but it needs a smarter system. Traditional databases struggle with complex data patterns, like images or text, which require "vector" representations. Vector databases excel here, storing and retrieving these patterns quickly and accurately.  

They scale effortlessly, handle real-time queries, and match user needs with precise results. This makes them vital for tools like chatbots, content creators, and recommendation systems. By bridging raw data and meaningful insights, vector databases enable faster, smarter AI responses. As AI evolves, they’ll remain key to unlocking deeper understanding and real-time decisions—making them a silent hero behind today’s most advanced tools. (149 words)

---
## Pattern 2 — Routing

**Concept:** A *classifier* LLM (or rule) reads the input and assigns it a category. A second LLM call — with a prompt specialised for that category — handles the actual response.

```
User Input → [Classifier LLM] → category → [Specialist LLM for category] → Response
```

**Use when:** you have distinct input types that benefit from different prompts, tools, or even different models.

**Example below:** Customer support triage — billing, technical, or general queries each get a tailored response.


In [2]:
SPECIALIST_PROMPTS = {
    "billing":   "You are a billing support specialist. Be empathetic and solution-focused.",
    "technical": "You are a senior technical support engineer. Give precise, step-by-step answers.",
    "general":   "You are a friendly customer success agent. Be warm and concise.",
}

def classify_query(user_query: str) -> str:
    """Return one of: billing | technical | general"""
    result = chat(
        [
            {"role": "system", "content": (
                "Classify the following customer query into exactly one category: "
                "billing, technical, or general. "
                "Reply with the single word only."
            )},
            {"role": "user", "content": user_query},
        ],
        temperature=0,
    ).strip().lower()
    # Normalise — model might add punctuation
    for key in SPECIALIST_PROMPTS:
        if key in result:
            return key
    return "general"

def routed_response(user_query: str) -> str:
    category = classify_query(user_query)
    system_prompt = SPECIALIST_PROMPTS[category]
    answer = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query},
    ])
    return category, answer

# ── Test three query types ────────────────────────────────────────────────────
queries = [
    "I was charged twice on my last invoice.",
    "My API requests keep returning a 429 error even though I'm below the limit.",
    "What are your business hours?",
]

for q in queries:
    category, answer = routed_response(q)
    display(Markdown(f"**Query:** {q}\n\n**Routed to:** `{category}`\n\n**Response:** {answer}\n\n---"))


**Query:** I was charged twice on my last invoice.

**Routed to:** `billing`

**Response:** I'm sorry to hear you encountered this issue—it’s frustrating to be charged twice, and I appreciate you bringing it to my attention. Let me help resolve this for you!  

To get started, could you share the **invoice number** and the **date** of the charges? This will help me locate your specific case quickly.  

Here’s what I’ll do:  
1. **Check your account** to see if there’s a duplicate charge or an error in processing.  
2. **Investigate** whether the second charge was a mistake (e.g., a system glitch or duplicate payment).  
3. **Resolve the issue** by either:  
   - **Refunding the duplicate amount** if it was an error, or  
   - **Providing a credit** if the charge was intentional but duplicated.  

Once I have the details, I’ll send you a clear explanation and next steps. This is a top priority for me, and I’ll keep you updated throughout the process.  

If you have any other questions or need further assistance, feel free to reach out. Thank you for your patience! 😊

---

**Query:** My API requests keep returning a 429 error even though I'm below the limit.

**Routed to:** `technical`

**Response:** A 429 "Too Many Requests" error occurs when your API calls exceed the rate limit. Even if you believe you're below the limit, here's how to diagnose and resolve the issue:

---

### **Step-by-Step Troubleshooting**

#### **1. Verify the Rate Limit Metrics**
   - **Check the API documentation** for specific rate limit rules (e.g., "100 requests/minute per IP", "500 requests/hour per API key").
   - **Identify the metric** being tracked (e.g., total requests, requests per endpoint, requests per IP, etc.).
   - **Compare your usage** against the documented limits.

#### **2. Confirm Your Request Count**
   - **Log all requests** (e.g., using a tool like Postman, curl, or a logging middleware) and count them.
   - Ensure you're **not counting non-API requests** (e.g., health checks, webhooks, or internal tools).

#### **3. Check Time Windows**
   - **Rate limits are often time-based** (e.g., "100 requests/minute"). Ensure your requests are within the same time window:
     - **UTC vs. local time**: Confirm the API uses UTC for rate limit calculations.
     - **Sliding window**: Some APIs use a "sliding window" (e.g., 100 requests in any 1-minute window), which can cause bursts to trigger limits.

#### **4. Validate Request Headers**
   - **API keys**: Ensure you're using the correct API key (e.g., different keys for different environments).
   - **User-Agent**: Some APIs throttle requests without a valid `User-Agent` header.
   - **Authentication**: Verify headers like `Authorization` or `X-API-Key` are correctly formatted.

#### **5. Check for Duplicate Requests**
   - **Idempotency keys**: If your API requires unique IDs for requests, ensure you're not reusing them.
   - **Caching**: If you're caching responses, ensure you're not making redundant requests (e.g., by reusing cached data without checking freshness).

#### **6. Test with a Minimal Tool**
   - Use **Postman** or **curl** to isolate the issue:
     ```bash
     curl -X GET https://api.example.com/data \
       -H "Authorization: Bearer YOUR_TOKEN" \
       -H "User-Agent: MyApp/1.0"
     ```
   - This avoids interference from libraries, proxies, or middleware.

#### **7. Check for Third-Party Interference**
   - **Proxies/load balancers**: If you're using a proxy, it might add headers or throttle traffic.
   - **CDNs**: Services like Cloudflare or Akamai might impose their own rate limits.

#### **8. Review API-Specific Limits**
   - Some APIs have **per-endpoint** limits (e.g., `/v1/data` might have a lower limit than `/v1/stats`).
   - Check if you're hitting a **sub-resource limit** (e.g., 100 requests/minute for `/v1/data`, but 500 for `/v1/data/123`).

#### **9. Use a Rate Limiting Tool**
   - Implement a **rate limiter** in your code (e.g., using Redis or a token bucket algorithm) to enforce limits on your side.

#### **10. Contact API Support**
   - If the issue persists, provide the following to the API provider:
     - **Request logs** (e.g., timestamps, endpoints, headers).
     - **Rate limit headers** (e.g., `X-RateLimit-Remaining`, `X-RateLimit-Reset`).
     - **Reproduction steps** (e.g., "I made 50 requests in 1 minute to `/v1/data`").

---

### **Example: Debugging with cURL**
```bash
curl -X GET "https://api.example.com/data" \
  -H "Authorization: Bearer YOUR_TOKEN" \
  -H "User-Agent: MyApp/1.0" \
  -v
```
- The `-v` flag shows detailed headers and response codes.
- Look for `X-RateLimit-*` headers in the response.

---

### **Prevention Tips**
- **Monitor usage** with tools like **Datadog**, **New Relic**, or **CloudWatch**.
- **Implement retries** with exponential backoff for transient 429 errors.
- **Use API gateways** (e.g., AWS API Gateway, Kong) to manage rate limits centrally.

Let me know your specific API or environment, and I can tailor the solution further!

---

**Query:** What are your business hours?

**Routed to:** `general`

**Response:** We're happy to help! Our business hours are **Monday to Friday, 9 AM to 6 PM** (local time). If you need assistance outside these hours, feel free to reach out—we’ll do our best to respond promptly! 😊 Let me know how I can assist further.

---

---
## Pattern 3 — Parallelization

**Concept:** Run multiple independent LLM calls at the same time and aggregate the results. Two sub-patterns:

- **Sectioning** — split a task into independent chunks (e.g. evaluate different aspects of code).
- **Voting** — run the same prompt N times and take a majority decision.

```
               ┌─ [Worker 1] ─┐
Input ─────────┤─ [Worker 2] ─├──► Aggregator ──► Final output
               └─ [Worker 3] ─┘
```

**Use when:** subtasks are independent, or you need higher confidence via multiple perspectives.

**Example below (Sectioning):** Evaluate a code snippet on three independent axes simultaneously — correctness, readability, security.


In [ ]:
CODE_SNIPPET = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query).fetchone()
"""

REVIEW_AXES = {
    "correctness":  "Review the code only for logical correctness and potential bugs.",
    "readability":  "Review the code only for readability, naming, and documentation.",
    "security":     "Review the code only for security vulnerabilities.",
}

def review_axis(axis: str, instructions: str) -> tuple[str, str]:
    result = chat([
        {"role": "system", "content": instructions + " Be concise (3–5 sentences)."},
        {"role": "user",   "content": f"```python\n{CODE_SNIPPET}\n```"},
    ])
    return axis, result

# ── Run reviews in parallel using threads ─────────────────────────────────────
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(review_axis, axis, instr): axis
               for axis, instr in REVIEW_AXES.items()}
    reviews = {}
    for future in futures:
        axis, result = future.result()
        reviews[axis] = result

# ── Aggregation step ──────────────────────────────────────────────────────────
review_text = "\n\n".join(f"### {k.title()}\n{v}" for k, v in reviews.items())
display(Markdown("## Parallel Code Reviews\n\n" + review_text))

summary = chat([
    {"role": "system", "content": "You are a lead code reviewer. Summarise the findings below into a single priority-ordered action list."},
    {"role": "user",   "content": review_text},
])
display(Markdown("## Aggregated Action List\n\n" + summary))


### Parallelization — Voting variant

Run the same classification N times and take a majority vote. Useful when a single call may be unreliable.


In [ ]:
from collections import Counter

REVIEW_TEXT = """
The new product update is amazing! I love how smooth everything feels now.
Although the onboarding is still a bit confusing, the core experience is 10/10.
"""

def classify_sentiment(_: int) -> str:
    """Classify sentiment as positive, negative, or mixed."""
    return chat(
        [
            {"role": "system", "content": "Classify the sentiment as exactly one word: positive, negative, or mixed."},
            {"role": "user",   "content": REVIEW_TEXT},
        ],
        temperature=1,   # higher temp → more variance across runs
    ).strip().lower().split()[0]

N_VOTES = 5
with ThreadPoolExecutor(max_workers=N_VOTES) as pool:
    votes = list(pool.map(classify_sentiment, range(N_VOTES)))

tally = Counter(votes)
winner = tally.most_common(1)[0][0]
display(Markdown(f"**Votes:** {dict(tally)}\n\n**Majority verdict:** `{winner}`"))


---
## Pattern 4 — Orchestrator-Workers

**Concept:** A central *orchestrator* LLM receives the goal, breaks it into subtasks dynamically, and delegates each subtask to a *worker* LLM. The orchestrator then synthesises all worker outputs.

```
Goal ──► [Orchestrator LLM] ──► subtask_1 ──► [Worker 1]
                             ├─► subtask_2 ──► [Worker 2]   ──► [Orchestrator synthesises]
                             └─► subtask_3 ──► [Worker 3]
```

**Key difference from Parallelization:** subtasks are *not pre-defined*; the orchestrator decides them at runtime.

**Use when:** the task is complex and the required subtasks aren't known in advance.

**Example below:** Research assistant — the orchestrator decides which sub-questions to investigate, workers answer each, orchestrator synthesises a final report.


In [ ]:
RESEARCH_GOAL = "Explain how Retrieval-Augmented Generation (RAG) works and when to use it."

# ── Orchestrator: break into subtasks ─────────────────────────────────────────
plan_raw = chat(
    [
        {"role": "system", "content": (
            "You are a research orchestrator. Given a goal, output a JSON array of 3–4 "
            "focused sub-questions to investigate. Return ONLY valid JSON, no other text."
        )},
        {"role": "user", "content": RESEARCH_GOAL},
    ],
    temperature=0,
)

# Parse JSON (strip markdown fences if the model adds them)
plan_clean = plan_raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
subtasks: list[str] = json.loads(plan_clean)
display(Markdown("### Orchestrator Plan\n\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(subtasks))))

# ── Workers: answer each subtask in parallel ──────────────────────────────────
def worker_answer(question: str) -> tuple[str, str]:
    answer = chat([
        {"role": "system", "content": "You are a concise AI/ML expert. Answer in 3–5 sentences."},
        {"role": "user",   "content": question},
    ])
    return question, answer

with ThreadPoolExecutor(max_workers=len(subtasks)) as pool:
    worker_results = dict(pool.map(lambda q: worker_answer(q), subtasks))

worker_text = "\n\n".join(f"**Q: {q}**\n\n{a}" for q, a in worker_results.items())
display(Markdown("### Worker Answers\n\n" + worker_text))

# ── Orchestrator: synthesise into a final report ──────────────────────────────
final_report = chat([
    {"role": "system", "content": "You are a senior technical writer. Synthesise the Q&A pairs below into a coherent, structured mini-report of ~200 words."},
    {"role": "user",   "content": f"Goal: {RESEARCH_GOAL}\n\n{worker_text}"},
])
display(Markdown("### Final Synthesised Report\n\n" + final_report))


---
## Pattern 5 — Evaluator-Optimizer

**Concept:** A *generator* LLM produces output; an *evaluator* LLM scores it and provides feedback; the generator uses that feedback to improve. The loop continues until quality criteria are met or a maximum iteration count is reached.

```
Task ──► [Generator] ──► draft
                ▲              │
           feedback      [Evaluator]
                ▲              │
                └──── score ◄──┘
           (loop until score ≥ threshold or max_iters)
```

**Use when:** you have clear quality criteria and iterative refinement measurably improves output. Analogous to a human writer → editor cycle.

**Example below:** Refine a technical explanation until the evaluator scores it ≥ 8/10 for clarity.


In [ ]:
WRITING_TASK = "Explain how attention mechanisms work in transformer models, for a junior developer."
SCORE_THRESHOLD = 8
MAX_ITERATIONS  = 4

def generate(task: str, feedback: str | None = None) -> str:
    messages = [
        {"role": "system", "content": "You are a technical educator. Write clearly and concisely (≤150 words)."},
        {"role": "user",   "content": task},
    ]
    if feedback:
        messages.append({"role": "user", "content": f"Improve your previous response based on this feedback:\n{feedback}"})
    return chat(messages)

def evaluate(task: str, response: str) -> tuple[int, str]:
    """Return (score 1-10, actionable feedback)."""
    raw = chat(
        [
            {"role": "system", "content": (
                "You are a strict writing evaluator. "
                "Score the response from 1–10 for clarity and accuracy given the task. "
                "Reply ONLY with JSON: {\"score\": <int>, \"feedback\": \"<string>\"}"
            )},
            {"role": "user", "content": f"Task: {task}\n\nResponse:\n{response}"},
        ],
        temperature=0,
    ).strip()
    clean = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    parsed = json.loads(clean)
    return int(parsed["score"]), parsed["feedback"]

# ── Optimization loop ─────────────────────────────────────────────────────────
feedback = None
for iteration in range(1, MAX_ITERATIONS + 1):
    draft = generate(WRITING_TASK, feedback)
    score, feedback = evaluate(WRITING_TASK, draft)

    display(Markdown(
        f"### Iteration {iteration} — Score: {score}/10\n\n"
        f"**Draft:**\n\n{draft}\n\n"
        f"**Evaluator feedback:** {feedback}"
    ))

    if score >= SCORE_THRESHOLD:
        display(Markdown(f"✅ **Threshold reached at iteration {iteration}. Final answer above.**"))
        break
else:
    display(Markdown(f"⚠️ Max iterations reached. Best score: {score}/10"))


---
## Pattern Comparison

Run the same task through **Prompt Chaining** vs **Evaluator-Optimizer** and compare quality vs latency trade-offs.


In [ ]:
import time

TASK = "Write a 100-word explanation of embeddings for a product manager."

# ── Prompt Chaining: outline → draft (2 steps, no feedback loop) ──────────────
t0 = time.time()
outline_pm = chat([
    {"role": "system", "content": "You are a technical content strategist."},
    {"role": "user",   "content": f"Write a 3-point outline for: {TASK}"},
])
draft_pm = chat([
    {"role": "system",    "content": "You are a technical writer for business audiences."},
    {"role": "assistant", "content": outline_pm},
    {"role": "user",      "content": "Expand into a 100-word explanation."},
])
chain_time = time.time() - t0

# ── Evaluator-Optimizer: 1 generation + 1 refinement pass ─────────────────────
t0 = time.time()
draft_eo = generate(TASK)
score_eo, fb_eo = evaluate(TASK, draft_eo)
if score_eo < 8:
    draft_eo = generate(TASK, fb_eo)
eo_time = time.time() - t0

display(Markdown(
    f"## Prompt Chaining result ({chain_time:.1f}s)\n\n{draft_pm}\n\n---\n\n"
    f"## Evaluator-Optimizer result ({eo_time:.1f}s) — initial score {score_eo}/10\n\n{draft_eo}"
))


---
## Exercises

Work through as many as you like. Start with (1) and go deeper as time permits.

1. **Prompt Chaining - add retry + structured gate:** In the blog pipeline (Pattern 1), enforce a strict outline format (exactly 3 numbered sections with titles only). If the gate fails, automatically re-prompt the model up to 2 retries before halting. Print which attempt passed.

2. **Routing — add a fourth category:** Extend the routing example to handle `"feature_request"` queries. Write a specialist prompt and test it with at least two example inputs.

3. **Parallelization — build a guardrail:** Create a parallel two-worker setup where Worker A generates a response to a user question while Worker B simultaneously checks whether the question is safe/appropriate. Combine the two results — only return Worker A's output if Worker B gives the green light.

4. **Orchestrator-Workers — dynamic depth:** Modify the orchestrator example so that after the first round of worker answers, the orchestrator decides whether a second round of deeper sub-questions is warranted. Implement at most 2 rounds.

5. **Evaluator-Optimizer — custom rubric:** Change the evaluator prompt to score on three separate axes (clarity, accuracy, brevity) and return a JSON object with all three scores. Stop the loop only when *all three* scores are ≥ 7.

6. **(Stretch) Combine patterns:** Build a mini content pipeline that uses:
   - **Routing** to detect if the topic is technical or business-oriented,
   - **Orchestrator-Workers** to research the topic in parallel, and
   - **Evaluator-Optimizer** to refine the final output.


---
## Exercise 1- Prompt Chaining with retry logic and structured gates

In [28]:
TOPIC = "Stock Investment for Dummies" 


def generate_outline_with_retries(topic, max_retries=2): 
    """Generate a strict 3-section outline with retry logic."""

    attempt = 0  
    outline = None
  
    while attempt <= max_retries: 
        attempt += 1
        outline = chat([  
            {"role": "system", "content": "You are a technical content strategist."},
            {"role": "user",   "content": f"Write an outline for a blog post about: {topic}. "
                                          "Format: exactly 3 numbered sections (1., 2., 3.) with titles only. "
                                          "No extra text, no subsections."},
        ]) 

        # Gate check: must have exactly 3 numbered sections – Once we get the outline, we validate it
        sections = re.findall(r'^\d+\.\s+.+$', outline, flags=re.MULTILINE)
        gate_pass = (len(sections) == 3)

        
        print(f"🔍 Attempt {attempt} — Gate pass: {gate_pass}") 

        if gate_pass:
            print(f"✅ Outline passed on attempt {attempt}")
            return outline
    
     
    raise ValueError("❌ Outline generation failed after retries.")

# ── Step 1: Generate outline with retry then, prints the outline in Markdown ────────────────────
outline = generate_outline_with_retries(TOPIC)
display(Markdown("### Step 1 — Outline\n\n" + outline))

# ── Step 2: Write the draft , expand the outline into 200-word draft. The persona here is senior technical writer. The outline is passed in as context, and the user asks for a concise draft then, prints the draft in Markdown─────────────────────────
draft = chat([
    {"role": "system",    "content": "You are a senior technical writer."},
    {"role": "user",      "content": f"Topic: {TOPIC}"},
    {"role": "assistant", "content": outline},
    {"role": "user",      "content": "Expand the outline into a concise 200-word blog post draft."},
])
display(Markdown("### Step 2 — Draft\n\n" + draft))

# ── Step 3: Polish for a non-technical audience. Finally, we polish the draft. The persona switches to editor, and the user prompt to rewrite the post for a non-technical audience, keeping it under 150 words. Then, prints the polished in Markdown───────────────────────────────
polished = chat([
    {"role": "system", "content": "You are an editor who simplifies technical writing."},
    {"role": "user",   "content": f"Rewrite the following blog post so it is engaging for a non-technical audience. "
                                  f"Keep it under 150 words.\n\n{draft}"},
])
display(Markdown("### Step 3 — Polished\n\n" + polished))


🔍 Attempt 1 — Gate pass: True
✅ Outline passed on attempt 1


### Step 1 — Outline

1. Understanding the Basics of Stock Investment  
2. Getting Started: How to Begin Investing in Stocks  
3. Common Mistakes to Avoid and Tips for Success

### Step 2 — Draft

**Stock Investment for Dummies: A Beginner’s Guide**  

Stock investment involves buying shares of a company, making you a partial owner. Companies use funds from stock sales to grow, and shareholders earn returns through dividends or rising stock prices. Understanding basics like market trends, dividends, and risk is key.  

To start, open a brokerage account, choose a platform, and set a budget. Begin with small investments, focusing on research—analyze company financials, industry trends, and growth potential. Diversify your portfolio to spread risk.  

Avoid common pitfalls: don’t chase hot stocks without research, avoid emotional decisions during market swings, and avoid overtrading. Instead, adopt a long-term mindset, reinvest dividends, and regularly review your strategy.  

Stocks can boost wealth, but success requires patience and knowledge. Start small, stay informed, and seek guidance to build confidence. With the right approach, investing becomes less intimidating and more rewarding. Remember: consistency and learning outweigh quick wins.

### Step 3 — Polished

**Investing in Stocks: A Simple Start for Newbies**  

Stocks let you own a piece of a company—like being a partner! When you buy shares, you help fund the company’s growth, and you can profit if the company succeeds through dividends or rising stock prices.  

To begin, open a brokerage account, pick a platform, and set a budget. Start small, research companies’ financial health, industries, and growth potential, then spread your investments to reduce risk.  

Avoid common traps: don’t chase hot stocks without checking the facts, avoid emotional decisions during market ups and downs, and don’t overtrade. Instead, think long-term, reinvest dividends, and review your plan regularly.  

Stocks can grow your wealth, but patience and learning matter most. Start small, stay curious, and seek help when needed. With time, investing becomes less intimidating—and more rewarding. Remember: consistency beats quick wins! (149 words)

---
## Exercise 2- Routing with additional "feature request" category

In [4]:
SPECIALIST_PROMPTS = {
    "billing":   "You are a billing support specialist. Be empathetic and solution-focused.",
    "technical": "You are a senior technical support engineer. Give precise, step-by-step answers.",
    "general":   "You are a friendly customer success agent. Be warm and concise.",
    "feature_request": "You are a Financial Manager. Evaluate feature requests in terms of cost, ROI, and budget impact. "
                       "Explain potential financial value and suggest prioritization based on resource allocation.",
}

def classify_query(user_query: str) -> str:
    """Return one of: billing | technical | general | feature_request"""
    result = chat(
        [
            {"role": "system", "content": (
                "Classify the following customer query into exactly one category: "
                "billing, technical, general, or feature_request. "
                "Reply with the single word only."
            )},
            {"role": "user", "content": user_query},
        ],
        temperature=0,
    ).strip().lower()
    # Normalize — model might add punctuation
    for key in SPECIALIST_PROMPTS:
        if key in result:
            return key
    return "general"

def routed_response(user_query: str):
    category = classify_query(user_query)
    system_prompt = SPECIALIST_PROMPTS[category]
    answer = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query},
    ])
    return category, answer

# ── Test queries ──────────────────────────────────────────────────────────────
queries = [
    "I was charged twice on my last invoice.",   # billing
    "My API requests keep returning a 429 error.", # technical
    "What are your business hours?",             # general
    "Can you add offline support for the mobile app?", # feature_request
    "I’d like a dark mode feature in the app.",  # feature_request
]

for q in queries:
    category, answer = routed_response(q)
    display(Markdown(f"**Query:** {q}\n\n**Routed to:** `{category}`\n\n**Response:** {answer}\n\n---"))

**Query:** I was charged twice on my last invoice.

**Routed to:** `billing`

**Response:** I'm sorry to hear that you were charged twice—it’s understandable to feel frustrated, and I appreciate you reaching out. Let’s resolve this together!  

To help speed up the process, could you share:  
1. The invoice number or the date of the duplicate charges?  
2. Any details about the service or product that was billed?  

Once I have this information, I’ll check the records, confirm if there’s a duplicate, and arrange for a refund or credit to your account. We’ll make sure this doesn’t happen again!  

If you’re unsure about the details, I’m happy to guide you through the steps. Thanks for your patience, and I’m here to help! 😊

---

**Query:** My API requests keep returning a 429 error.

**Routed to:** `technical`

**Response:** A **429 Too Many Requests** error occurs when your API client exceeds the server's rate-limiting thresholds. Here's a step-by-step guide to resolve this:

---

### **1. Check Your Request Rate**
- **Count requests**: Verify how many requests your client is sending per second/minute. Use tools like:
  - **Wireshark** or **tcpdump** (network analysis).
  - **Postman** or **curl** (manual testing).
  - **Application logs** (if your app logs request counts).
- **Compare to limits**: Ensure your request rate is below the API's documented limits.

---

### **2. Review API Rate-Limiting Policies**
- **Check documentation**: Look for rate-limit headers like:
  - `X-RateLimit-Limit` (max allowed per window).
  - `X-RateLimit-Remaining` (remaining requests).
  - `X-RateLimit-Reset` (time until reset).
- **Example headers**:
  ```http
  HTTP/1.1 429 Too Many Requests
  Content-Type: application/json
  Retry-After: 60
  ```

---

### **3. Implement Retry Logic with Exponential Backoff**
- **Add retries**: Use a retry mechanism with delays to avoid overwhelming the server.
- **Example (Python)**:
  ```python
  import time
  import requests

  for attempt in range(5):
      try:
          response = requests.get("https://api.example.com/data")
          response.raise_for_status()
          break
      except requests.HTTPError as e:
          if e.response.status_code == 429:
              wait = 2 ** attempt  # Exponential backoff
              print(f"429: Retrying in {wait} seconds...")
              time.sleep(wait)
          else:
              raise
  ```

---

### **4. Use a Rate-Limiting Client Library**
- **Leverage libraries** like `ratelimit` (Python) or `axios` (JavaScript) to enforce limits on your client side.

---

### **5. Optimize Request Patterns**
- **Batch requests**: Combine multiple queries into a single request (e.g., bulk endpoints).
- **Cache responses**: Store results locally to reduce redundant requests.
- **Use pagination**: Avoid fetching large datasets in one go.

---

### **6. Contact the API Provider**
- **Reach out to support**: If the issue persists, provide:
  - Your IP address (if applicable).
  - Request frequency and timestamps.
  - Example headers from the 429 response.
- **Ask for**: 
  - Current rate-limit tiers.
  - Options to upgrade your plan (if applicable).

---

### **7. Monitor with Tools**
- **Use tools** like:
  - **Postman** (to test request rates).
  - **CloudWatch** or **Datadog** (monitor API usage).
  - **API gateways** (e.g., AWS API Gateway) to track quotas.

---

### **Key Headers to Check**
- `Retry-After`: Indicates how long to wait before retrying (e.g., `Retry-After: 60`).
- `X-RateLimit-Remaining`: Shows how many requests are left in the current window.

---

By systematically addressing these areas, you should resolve the 429 error. If the problem persists, provide specific details (e.g., API name, headers, request frequency) for further assistance.

---

**Query:** What are your business hours?

**Routed to:** `general`

**Response:** We're here to help! Our business hours are Monday to Friday, 9:00 AM to 6:00 PM. For urgent assistance outside these hours, feel free to reach out via email or chat—we'll do our best to support you! 😊

---

**Query:** Can you add offline support for the mobile app?

**Routed to:** `feature_request`

**Response:** **Evaluation of Adding Offline Support for the Mobile App**

**1. Development Cost:**  
- **Estimated Cost:** $15,000–$30,000 (depending on app complexity, platform (iOS/Android), and third-party tools).  
- **Key Considerations:**  
  - Local storage implementation (e.g., SQLite, Realm, or Firebase).  
  - Data synchronization logic (e.g., conflict resolution, delta sync).  
  - Backend updates to handle offline data queues.  
  - Testing for edge cases (e.g., intermittent connectivity, data integrity).  

**2. Return on Investment (ROI):**  
- **User Retention:** Improved offline support could increase user engagement by 10–15% in low-connectivity regions (e.g., rural areas, travel).  
- **Revenue Impact:**  
  - Higher retention = increased lifetime value (LTV) of users.  
  - Reduced server costs from minimized real-time sync (e.g., 20–30% reduction in API calls).  
- **Market Differentiation:** If competitors lack this feature, it could improve user acquisition and satisfaction.  

**3. Budget Impact:**  
- **Resource Allocation:** This request would require reallocating budget from other features (e.g., UX improvements, backend scalability).  
- **Opportunity Cost:** Compare with higher-impact priorities (e.g., a feature that directly drives monetization or reduces churn).  

**4. Financial Value Potential:**  
- **Direct Savings:** $5,000–$10,000 annually in reduced server costs (if 10% of users use offline mode).  
- **Indirect Revenue:** Potential 10–15% increase in active users could translate to $50,000–$150,000 in additional revenue (depending on monetization model).  

**5. Prioritization Recommendations:**  
- **Short-Term:**  
  - **Phase 1:** Implement core offline functionality (e.g., basic data caching) with minimal sync logic to test user adoption.  
  - **Budget:** Allocate $10,000 for MVP development.  
- **Long-Term:**  
  - If MVP shows positive ROI, scale to full offline support with advanced sync features.  
  - **Alternative:** If budget is constrained, prioritize features with higher immediate revenue impact (e.g., subscription upgrades or ad optimization).  

**Conclusion:**  
Adding offline support is a **high-impact, medium-cost initiative** that balances user retention and operational savings. Prioritize it if:  
- Your user base includes low-connectivity regions.  
- Competitors lack this feature.  
- You can test the MVP with limited resources to validate ROI before full-scale implementation.  

**Next Steps:**  
1. Conduct a cost-benefit analysis with the development team.  
2. Run a pilot with a limited feature set to assess user adoption.  
3. Compare with other initiatives to ensure alignment with strategic goals.

---

**Query:** I’d like a dark mode feature in the app.

**Routed to:** `feature_request`

**Response:** **Evaluation of Dark Mode Feature Request**  
**1. Cost Analysis**  
- **Development Cost**:  
  - **Low to Moderate**: If the app uses frameworks like Flutter, React Native, or SwiftUI, dark mode can often be implemented with minimal code changes (e.g., theme toggling). However, if the UI requires significant redesign (e.g., custom components), costs may rise.  
  - **Testing**: Additional time is needed to ensure compatibility across devices, OS versions, and screen types (e.g., OLED vs. LCD).  

- **Maintenance Cost**:  
  - Dark mode may require ongoing updates to ensure consistency with new OS features (e.g., Android 10+ system-wide dark mode).  

**2. ROI and Financial Value**  
- **User Retention & Satisfaction**:  
  - Dark mode improves accessibility for users in low-light environments and reduces eye strain, potentially increasing user engagement and retention. Studies suggest users prefer dark interfaces for extended use.  
  - **Competitive Advantage**: In markets where dark mode is standard (e.g., social media, productivity tools), lacking it could reduce user adoption.  

- **Energy Efficiency**:  
  - On OLED screens, dark mode reduces power consumption, which may appeal to eco-conscious users and indirectly support brand reputation.  

- **Monetary Impact**:  
  - While direct revenue may not be immediate, improved retention and reduced churn could enhance long-term profitability.  

**3. Budget Impact**  
- **Short-Term**: A low-cost implementation (e.g., theme toggle) is feasible within existing budgets.  
- **Long-Term**: If the app is in a mature phase with limited development capacity, prioritizing high-impact features (e.g., core functionality enhancements) may be more cost-effective.  

**4. Prioritization Recommendations**  
- **High Priority**: If:  
  - The app targets users in low-light environments (e.g., night owls, outdoor workers).  
  - Competitors already offer dark mode, and its absence risks losing market share.  
  - The team has spare capacity to implement it without delaying critical features.  

- **Medium Priority**: If:  
  - The app is in a growth phase, and dark mode aligns with UX improvements but isn’t critical for retention.  
  - The team lacks resources, and the cost-benefit analysis is marginal.  

- **Low Priority**: If:  
  - The app has a small user base with no demand for dark mode.  
  - The cost of implementation exceeds potential ROI (e.g., limited user engagement metrics).  

**5. Suggested Approach**  
- **Pilot Implementation**: Start with a low-cost, incremental rollout (e.g., theme toggle) to test user adoption and gather data on retention metrics.  
- **A/B Testing**: Compare user engagement (e.g., session duration, churn rate) between dark and light modes to quantify ROI.  
- **Budget Allocation**: If approved, allocate resources to ensure cross-platform compatibility and user onboarding for the new feature.  

**Conclusion**: Dark mode is a low-cost, high-value feature with potential to enhance user experience and retention. Prioritize it if it aligns with user needs, competitive benchmarks, and available resources. If budget constraints exist, pair it with other low-cost UX improvements for maximum impact.

---

In [5]:
SPECIALIST_PROMPTS = {
    "billing":   "You are a billing support specialist. Be empathetic and solution-focused.",
    "technical": "You are a senior technical support engineer. Give precise, step-by-step answers.",
    "general":   "You are a friendly customer success agent. Be warm and concise.",
    "feature_request": "You are a Financial Manager. Evaluate feature requests in terms of cost, ROI, and budget impact. "
                       "Explain potential financial value and suggest prioritization based on resource allocation.",
}

def classify_query(user_query: str) -> str:
    """Return one of: billing | technical | general | feature_request"""
    result = chat(
        [
            {"role": "system", "content": (
                "Classify the following customer query into exactly one category: "
                "billing, technical, general, or feature_request. "
                "Reply with the single word only."
            )},
            {"role": "user", "content": user_query},
        ],
        temperature=0,
    ).strip().lower()
    # Normalize — model might add punctuation
    for key in SPECIALIST_PROMPTS:
        if key in result:
            return key
    return "general"

def routed_response(user_query: str):
    category = classify_query(user_query)
    system_prompt = SPECIALIST_PROMPTS[category]
    answer = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_query},
    ])
    return category, answer

# ── Unit Tests ────────────────────────────────────────────────────────────────
def test_routing():
    # Billing query
    q1 = "I was charged twice on my last invoice."
    assert classify_query(q1) == "billing"

    # Technical query
    q2 = "My API requests keep returning a 429 error."
    assert classify_query(q2) == "technical"

    # General query
    q3 = "What are your business hours?"
    assert classify_query(q3) == "general"

    # Feature request queries
    q4 = "Can you add offline support for the mobile app?"
    assert classify_query(q4) == "feature_request"

    q5 = "I’d like a dark mode feature in the app."
    assert classify_query(q5) == "feature_request"

    print("✅ All routing tests passed!")

# Run the tests
test_routing()

✅ All routing tests passed!


---
## Exercise 3- Parallelization

In [6]:

def worker_a_generate(user_query: str) -> str:
    """Worker A: Generate a response to the user query."""
    response = chat([
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_query},
    ])
    return response

def worker_b_guardrail(user_query: str) -> bool:
    """Worker B: Check if the query is safe/appropriate."""
    verdict = chat([
        {"role": "system", "content": (
            "Decide if the following query is safe and appropriate. "
            "Reply only with 'safe' or 'unsafe'."
        )},
        {"role": "user", "content": user_query},
    ]).strip().lower()
    return verdict == "safe"

def guarded_response(user_query: str):
    """Run Worker A and Worker B in parallel, return A's output only if B approves."""
    with ThreadPoolExecutor() as executor:
        future_a = executor.submit(worker_a_generate, user_query)
        future_b = executor.submit(worker_b_guardrail, user_query)
        response_a = future_a.result()
        verdict_b = future_b.result()
    if verdict_b:
        return f"✅ Safe. Response:\n{response_a}"
    else:
        return "❌ Query deemed unsafe. No response generated."

# ── Example usage ──────────────────────────────────────────────────────────
queries = [
    "Can you explain how vector databases help retrieval-augmented generation?",
    "Tell me how to hack into someone's account.",  # unsafe
]

for q in queries:
    print(f"User query: {q}")
    print(guarded_response(q))
    print("---")


User query: Can you explain how vector databases help retrieval-augmented generation?
✅ Safe. Response:
Vector databases play a crucial role in **retrieval-augmented generation (RAG)** by enabling efficient and accurate retrieval of relevant information from large, unstructured datasets. Here's how they help:

---

### **1. Semantic Understanding via Vector Embeddings**
- **How it works**: Text (or other data) is converted into numerical vectors (embeddings) using models like BERT, Sentence-BERT, or other language models. These vectors capture the **semantic meaning** of the text, not just keyword matches.
- **Impact on RAG**: When a query is input into a RAG system, it is first converted into a vector. The vector database then searches for **similar vectors** (i.e., semantically relevant documents) from its stored embeddings. This allows the system to retrieve contextually relevant information, even if the query doesn’t use exact keywords.

---

### **2. Efficient Retrieval of Large D

---
## Exercise 4- Orchestration-Workers


In [7]:

RESEARCH_GOAL = "Explain how Stock Investment works and when to usedive right into it."

# ── Utility: clean JSON output from model ─────────────────────────────────────
def clean_json(raw: str) -> str:
    text = raw.strip()
    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    if text.startswith("```"):
        text = text[len("```"):].strip()
    if text.endswith("```"):
        text = text[:-3].strip()
    return text

# ── Orchestrator: generate subtasks ──────────────────────────────────────────
def orchestrator_plan(goal: str, context: str = "", max_retries: int = 2) -> list[str]:
    """Generate subtasks based on the goal and optional context (previous answers)."""
    system_prompt = (
        "You are a research orchestrator. Given a goal, output ONLY a JSON array of 3–4 "
        "focused sub-questions (strings). Do not add explanations or text outside JSON."
    )
    if context:
        user_prompt = f"Goal: {goal}\n\nContext from previous answers:\n{context}"
    else:
        user_prompt = goal

    for attempt in range(max_retries):
        plan_raw = chat([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ], temperature=0)

        plan_clean = clean_json(plan_raw)
        try:
            subtasks = json.loads(plan_clean)
            if isinstance(subtasks, list) and all(isinstance(t, str) for t in subtasks):
                return subtasks
        except json.JSONDecodeError:
            print(f"⚠️ Invalid JSON on attempt {attempt+1}, retrying...")

    # Fallback if all retries fail
    print("⚠️ Using fallback subtasks.")
    return [
        f"What is {goal}?",
        f"How does {goal} work in practice?",
        f"When should {goal} be applied?",
    ]

# ── Worker: answer a subtask ─────────────────────────────────────────────────
def worker_answer(question: str) -> tuple[str, str]:
    answer = chat([
        {"role": "system", "content": "You are a concise AI/ML expert. Answer in 3–5 sentences."},
        {"role": "user",   "content": question},
    ])
    return question, answer

# ── Orchestrator: decide if deeper round is needed ───────────────────────────
def orchestrator_decide(goal: str, worker_text: str) -> bool:
    decision = chat([
        {"role": "system", "content": (
            "You are a research orchestrator. Given the goal and the first round of answers, "
            "decide if a second round of deeper sub-questions is warranted. "
            "Reply ONLY with 'yes' or 'no'."
        )},
        {"role": "user", "content": f"Goal: {goal}\n\nAnswers:\n{worker_text}"},
    ], temperature=0).strip().lower()
    return decision == "yes"

# ── Pipeline ─────────────────────────────────────────────────────────────────
rounds = []
goal = RESEARCH_GOAL
context = ""

for depth in range(1, 3):  # at most 2 rounds
    try:
        subtasks = orchestrator_plan(goal, context)
    except ValueError as e:
        print(f"⚠️ Orchestrator failed to generate subtasks: {e}")
        break

    if not subtasks:
        print("⚠️ No subtasks generated, stopping pipeline.")
        break

    # Numbering starts at 1 for readability
    display(Markdown(
        f"### Round {depth} — Orchestrator Plan\n\n" +
        "\n".join(f"{i}. {t}" for i, t in enumerate(subtasks, start=1))
    ))

    # Run workers in parallel
    with ThreadPoolExecutor(max_workers=len(subtasks)) as pool:
        worker_results = list(pool.map(worker_answer, subtasks))

    # Build worker answers text
    worker_text = "\n\n".join(f"**Q: {q}**\n\n{a}" for q, a in worker_results)
    display(Markdown(f"### Round {depth} — Worker Answers\n\n" + worker_text))

    rounds.append(worker_text)

    # Decision logic: after first round, orchestrator decides if deeper round is needed
    if depth == 1:
        if not orchestrator_decide(goal, worker_text):
            break
        else:
            # Pass previous answers as context for deeper subtasks
            context = worker_text

# ── Final synthesis ─────────────────────────────────────────────────────────
combined_text = "\n\n".join(rounds)
final_report = chat([
    {"role": "system", "content": (
        "You are a senior technical writer. Synthesise the Q&A pairs below into a coherent, "
        "structured mini-report of ~200 words."
    )},
    {"role": "user", "content": f"Goal: {RESEARCH_GOAL}\n\n{combined_text}"},
])
display(Markdown("### Final Synthesised Report\n\n" + final_report))


### Round 1 — Orchestrator Plan

1. What are the fundamental mechanics of stock investment, including buying/selling shares and market dynamics?
2. What factors determine the optimal timing to start investing in stocks?
3. How do risk management strategies influence decision-making when entering the stock market?
4. What criteria should an individual use to assess their readiness to invest in stocks?

### Round 1 — Worker Answers

**Q: What are the fundamental mechanics of stock investment, including buying/selling shares and market dynamics?**

Stock investment involves buying/selling shares of companies through exchanges, where prices are determined by supply-demand dynamics. Investors purchase shares via brokers, owning a portion of the company’s equity, and sell them to transfer ownership. Market dynamics are driven by factors like company performance, economic indicators, and investor sentiment, causing price fluctuations. Liquidity and trading volume also influence how easily shares can be bought/sold. Long-term growth potential and dividends often motivate investors, though risks like volatility and market downturns are inherent.

**Q: What factors determine the optimal timing to start investing in stocks?**

The optimal timing to start investing in stocks depends on market conditions, personal financial goals, risk tolerance, and long-term strategy. While timing the market is inherently challenging, starting early allows compounding growth. Assess your financial stability, emergency fund, and ability to withstand volatility. Dollar-cost averaging can mitigate timing risks by spreading investments over time. Economic indicators and market trends may influence decisions, but consistent, informed investing often outperforms trying to predict exact market peaks.

**Q: How do risk management strategies influence decision-making when entering the stock market?**

Risk management strategies influence stock market decision-making by prioritizing capital preservation and aligning investments with an investor's risk tolerance. Techniques like diversification, position sizing, and stop-loss orders help quantify risk exposure, guiding choices on asset allocation and entry points. These strategies also shape trade execution, such as using hedging tools (e.g., options) to mitigate downside risk. Ultimately, they balance potential returns with acceptable volatility, ensuring decisions reflect both financial goals and risk appetite.

**Q: What criteria should an individual use to assess their readiness to invest in stocks?**

To assess readiness for stock investing, evaluate your financial goals, risk tolerance, and time horizon. Ensure you have an emergency fund to avoid panic selling during market downturns. Understand basic market mechanics and diversify to mitigate risk. Practice disciplined investing, avoiding emotional decisions, and align your strategy with long-term objectives. Finally, consider consulting a financial advisor if unsure about your approach.

### Final Synthesised Report

**Understanding Stock Investment: Mechanics, Timing, and Readiness**  

Stock investment involves purchasing shares of companies through exchanges, where prices are shaped by supply-demand dynamics, company performance, economic indicators, and investor sentiment. Investors own a portion of a company’s equity, with potential returns from capital gains and dividends. However, risks like market volatility and liquidity challenges are inherent.  

Optimal entry timing depends on market conditions, personal financial goals, and risk tolerance. While timing the market is difficult, starting early leverages compounding growth. Assess financial stability, emergency funds, and ability to withstand volatility. Dollar-cost averaging can mitigate timing risks by spreading investments over time.  

Risk management is critical, with strategies like diversification, position sizing, and stop-loss orders helping balance returns and volatility. Hedging tools, such as options, further protect against downside risks.  

Readiness to invest hinges on financial goals, risk tolerance, and time horizon. Ensure an emergency fund, understand market mechanics, and prioritize disciplined, diversified strategies. Avoid emotional decisions and align investments with long-term objectives. Consulting a financial advisor can provide tailored guidance.  

In summary, stock investing requires understanding mechanics, strategic timing, proactive risk management, and personal preparedness to navigate market uncertainties effectively.

---
## Exercise 5- Evaluator-Optimizer

In [8]:

WRITING_TASK    = "Explain how attention mechanisms work in transformer models, for a junior developer."
SCORE_THRESHOLD = 7
MAX_ITERATIONS  = 4

# ── Generator: produce or refine a draft ──────────────────────────────────────
def generate(task: str, feedback: str | None = None) -> str:
    messages = [
        {"role": "system", "content": "You are a technical educator. Write clearly and concisely (≤150 words)."},
        {"role": "user",   "content": task},
    ]
    if feedback:
        messages.append({"role": "user", "content": f"Improve your previous response based on this feedback:\n{feedback}"})
    return chat(messages)

# ── Evaluator: score on three axes ───────────────────────────────────────────
def evaluate(task: str, response: str) -> tuple[dict, str]:
    """
    Return ({'clarity': int, 'accuracy': int, 'brevity': int}, feedback string).
    """
    raw = chat(
        [
            {"role": "system", "content": (
                "You are a strict writing evaluator. "
                "Score the response from 1–10 on three axes: clarity, accuracy, brevity. "
                "Reply ONLY with JSON: {\"clarity\": <int>, \"accuracy\": <int>, \"brevity\": <int>, \"feedback\": \"<string>\"}"
            )},
            {"role": "user", "content": f"Task: {task}\n\nResponse:\n{response}"},
        ],
        temperature=0,
    ).strip()
    clean = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    parsed = json.loads(clean)
    scores = {axis: int(parsed[axis]) for axis in ["clarity", "accuracy", "brevity"]}
    return scores, parsed["feedback"]

# ── Optimization loop ─────────────────────────────────────────────────────────
feedback = None
for iteration in range(1, MAX_ITERATIONS + 1):
    draft = generate(WRITING_TASK, feedback)
    scores, feedback = evaluate(WRITING_TASK, draft)

    display(Markdown(
        f"### Iteration {iteration}\n"
        f"Scores → Clarity: {scores['clarity']}/10, Accuracy: {scores['accuracy']}/10, Brevity: {scores['brevity']}/10\n\n"
        f"**Draft:**\n\n{draft}\n\n"
        f"**Evaluator feedback:** {feedback}"
    ))

    # Check if all three scores meet threshold
    if all(score >= SCORE_THRESHOLD for score in scores.values()):
        display(Markdown(f"✅ **Threshold reached at iteration {iteration}. Final answer above.**"))
        break
else:
    display(Markdown(
        f"⚠️ Max iterations reached. Best scores → "
        f"Clarity: {scores['clarity']}, Accuracy: {scores['accuracy']}, Brevity: {scores['brevity']}"
    ))

### Iteration 1
Scores → Clarity: 9/10, Accuracy: 8/10, Brevity: 9/10

**Draft:**

Attention mechanisms in transformers let the model focus on relevant parts of input data when processing each element. Imagine reading a sentence: instead of treating every word equally, the model prioritizes words that matter most for the current task (e.g., translation or prediction).  

Here’s how it works:  
1. **Queries, Keys, Values**: For each word, the model creates three representations: a *query* (what it’s looking for), a *key* (what it’s comparing), and a *value* (the actual data).  
2. **Attention Scores**: The model calculates how well each *query* matches every *key*, producing scores. Higher scores mean more focus.  
3. **Weights and Context**: Scores are normalized using softmax, then multiplied by *values* to create a weighted context vector—this captures the most relevant information for the current word.  

This lets transformers dynamically weigh input parts, improving tasks like translation or text generation by focusing on key relationships. Simple, right? 😊

**Evaluator feedback:** The explanation is clear and uses relatable analogies. It accurately describes the core components (queries/keys/values) and process. Brevity is excellent. Slight improvement could be made by briefly mentioning how the weighted context is used in the final output, but overall this is a strong explanation for a junior developer.

✅ **Threshold reached at iteration 1. Final answer above.**

---
## Exercise 6- Combined Patterns